<a href="https://colab.research.google.com/github/NourahAlmutairi-max/predictive-maintenance-machine-learning/blob/main/predictive_maintenance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predictive Maintenance Using Machine Learning

## Project Overview

This project develops a machine learning-based predictive maintenance system for equipment failure prediction and condition monitoring.

The system uses sensor and operational data to:
- Predict machine failures using a Random Forest classifier.
- Address class imbalance using class weighting.
- Optimize the classification threshold using a validation set.
- Detect unusual equipment behavior using Isolation Forest.
- Explain model predictions using SHAP.
- Generate structured results for an interactive Power BI dashboard.

The final Random Forest model achieved:

- **Accuracy:** 97.93%
- **Precision:** 70.83%
- **Recall:** 66.67%
- **F1 Score:** 68.69%
- **PR-AUC:** 0.761

Because machine failures represent a small minority of the dataset, Precision, Recall, F1 Score, and PR-AUC were used alongside Accuracy to evaluate model performance.

## Dataset and Features

The project uses the AI4I 2020 Predictive Maintenance Dataset containing 10,000 equipment records.

Five operational features are used for machine failure prediction:

- Air Temperature [K]
- Process Temperature [K]
- Rotational Speed [rpm]
- Torque [Nm]
- Tool Wear [min]

The target variable is **Machine Failure**, where:

- `0` = Normal operation
- `1` = Machine failure

The dataset is highly imbalanced, with substantially fewer failure records than normal operating records.

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("/ai4i2020.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## Train, Validation, and Test Split

The dataset was divided into three subsets using stratified sampling:

- **70% Training Set** — used to train the machine learning model.
- **15% Validation Set** — used to optimize the classification threshold.
- **15% Test Set** — used only for the final model evaluation.

Stratification was used to preserve the class distribution across all subsets and reduce evaluation bias.

In [5]:
from sklearn.model_selection import train_test_split

# Same 5 sensor features used in the original project
features = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

target = "Machine failure"

X = df[features]
y = df[target]

# 70% Train, 15% Validation, 15% Test
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

print("\nFailure counts:")
print("Train:", y_train.value_counts().to_dict())
print("Validation:", y_val.value_counts().to_dict())
print("Test:", y_test.value_counts().to_dict())

Train: (7000, 5) (7000,)
Validation: (1500, 5) (1500,)
Test: (1500, 5) (1500,)

Failure counts:
Train: {0: 6763, 1: 237}
Validation: {0: 1449, 1: 51}
Test: {0: 1449, 1: 51}


## Random Forest Model

A Random Forest classifier was trained to predict machine failures from the five operational features.

The model uses:

- **200 decision trees**
- **Class weighting** to address the imbalance between normal and failure cases
- A fixed **random state** for reproducibility

Random Forest was selected as the final classification model for its ability to capture nonlinear relationships between equipment variables while providing robust performance on tabular data.

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    average_precision_score
)

# Build Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Train
rf_model.fit(X_train, y_train)

# Validation probabilities
rf_val_prob = rf_model.predict_proba(X_val)[:, 1]

print("Random Forest trained successfully.")

Random Forest trained successfully.


## Classification Threshold Optimization

Instead of automatically using the default probability threshold of 0.50, multiple classification thresholds were evaluated on the validation set.

The tested thresholds were:

- 0.50
- 0.40
- 0.30
- 0.20

The threshold with the highest validation **F1 Score** was selected.

The final selected threshold was **0.30**, providing a better balance between Precision and Recall for the minority failure class.

Importantly, the test set was not used during threshold selection and was reserved for final evaluation.

In [7]:
thresholds = [0.50, 0.40, 0.30, 0.20]

best_rf_threshold = None
best_rf_f1 = -1

print("Random Forest - Validation Results\n")

for threshold in thresholds:
    predictions = (rf_val_prob >= threshold).astype(int)

    precision = precision_score(y_val, predictions, zero_division=0)
    recall = recall_score(y_val, predictions, zero_division=0)
    f1 = f1_score(y_val, predictions, zero_division=0)

    print(
        f"Threshold {threshold:.2f} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | "
        f"F1: {f1:.4f}"
    )

    if f1 > best_rf_f1:
        best_rf_f1 = f1
        best_rf_threshold = threshold

print("\nBest RF Threshold:", best_rf_threshold)
print("Best Validation F1:", round(best_rf_f1, 4))

Random Forest - Validation Results

Threshold 0.50 | Precision: 0.7778 | Recall: 0.5490 | F1: 0.6437
Threshold 0.40 | Precision: 0.7619 | Recall: 0.6275 | F1: 0.6882
Threshold 0.30 | Precision: 0.7115 | Recall: 0.7255 | F1: 0.7184
Threshold 0.20 | Precision: 0.5857 | Recall: 0.8039 | F1: 0.6777

Best RF Threshold: 0.3
Best Validation F1: 0.7184


## Final Model Evaluation

The optimized Random Forest model was evaluated on the held-out test set using the classification threshold selected from the validation set.

The final test results were:

- **Accuracy:** 97.93%
- **Precision:** 70.83%
- **Recall:** 66.67%
- **F1 Score:** 68.69%
- **PR-AUC:** 0.761

The confusion matrix showed:

- **True Negatives:** 1,435
- **False Positives:** 14
- **False Negatives:** 17
- **True Positives:** 34

Because the dataset is highly imbalanced, Accuracy alone can be misleading. Therefore, Precision, Recall, F1 Score, and PR-AUC were used to provide a more informative evaluation of the model's ability to identify machine failures.

In [13]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

# --------------------------------------------------
# Random Forest - Final Test Evaluation
# --------------------------------------------------

rf_test_prob = rf_model.predict_proba(X_test)[:, 1]
rf_test_pred = (rf_test_prob >= best_rf_threshold).astype(int)

rf_accuracy = accuracy_score(y_test, rf_test_pred)
rf_precision = precision_score(y_test, rf_test_pred, zero_division=0)
rf_recall = recall_score(y_test, rf_test_pred, zero_division=0)
rf_f1 = f1_score(y_test, rf_test_pred, zero_division=0)
rf_pr_auc = average_precision_score(y_test, rf_test_prob)

print("=" * 55)
print("RANDOM FOREST - FINAL TEST RESULTS")
print("=" * 55)

print(f"Selected Threshold: {best_rf_threshold:.2f}")
print(f"Accuracy:  {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall:    {rf_recall:.4f}")
print(f"F1 Score:  {rf_f1:.4f}")
print(f"PR-AUC:    {rf_pr_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_test_pred))

RANDOM FOREST - FINAL TEST RESULTS
Selected Threshold: 0.30
Accuracy:  0.9793
Precision: 0.7083
Recall:    0.6667
F1 Score:  0.6869
PR-AUC:    0.7606

Confusion Matrix:
[[1435   14]
 [  17   34]]


## Anomaly Detection with Isolation Forest

Isolation Forest was used as an unsupervised anomaly detection method to identify unusual equipment operating conditions without using the failure labels during model training.

The model was trained on the operational sensor features and then applied to the test set.

The analysis identified:

- **318 anomalous records**
- **1,182 normal records**
- **Anomaly rate:** 21.20%

The observed failure rate among anomalous records was approximately **11.95%**, compared with approximately **1.10%** among records classified as normal.

These results indicate that unusual operating patterns identified by Isolation Forest were associated with a higher observed failure rate. However, anomaly detection does not directly predict or prove the cause of equipment failure.

In [18]:
from sklearn.ensemble import IsolationForest
import numpy as np

# --------------------------------------------------
# Isolation Forest - Anomaly Detection
# --------------------------------------------------

# Set the expected anomaly proportion for the Isolation Forest model
# as an experimental parameter for anomaly detection
isolation_model = IsolationForest(
    contamination=0.22,
    random_state=42
)

# Train using sensor features only
isolation_model.fit(X_train)

# Detect anomalies in the test set
anomaly_predictions = isolation_model.predict(X_test)
anomaly_scores = isolation_model.decision_function(X_test)

# Isolation Forest output:
#  1 = Normal
# -1 = Anomaly

anomaly_status = np.where(
    anomaly_predictions == -1,
    "Anomaly",
    "Normal"
)

# --------------------------------------------------
# Create anomaly results table
# --------------------------------------------------

anomaly_results = X_test.copy()

anomaly_results["Actual Failure"] = y_test.values
anomaly_results["Anomaly Status"] = anomaly_status
anomaly_results["Anomaly Score"] = anomaly_scores

# --------------------------------------------------
# Display results
# --------------------------------------------------

print("=" * 55)
print("ISOLATION FOREST - ANOMALY DETECTION RESULTS")
print("=" * 55)

print("\nAnomaly Distribution:")
print(anomaly_results["Anomaly Status"].value_counts())

print("\nAnomaly Rate:")
anomaly_rate = (
    anomaly_results["Anomaly Status"].eq("Anomaly").mean()
)

print(f"{anomaly_rate:.2%}")

print("\nFailure Rate by Anomaly Status:")
print(
    anomaly_results.groupby("Anomaly Status")["Actual Failure"]
    .agg(["count", "sum", "mean"])
)

ISOLATION FOREST - ANOMALY DETECTION RESULTS

Anomaly Distribution:
Anomaly Status
Normal     1182
Anomaly     318
Name: count, dtype: int64

Anomaly Rate:
21.20%

Failure Rate by Anomaly Status:
                count  sum      mean
Anomaly Status                      
Anomaly           318   38  0.119497
Normal           1182   13  0.010998


## Explainable AI with SHAP

SHAP (SHapley Additive exPlanations) was used to improve model interpretability by measuring how each feature contributes to the Random Forest predictions.

The global SHAP analysis ranked the features by their average absolute contribution:

1. **Torque [Nm]** — 0.1860
2. **Rotational Speed [rpm]** — 0.1317
3. **Tool Wear [min]** — 0.0977
4. **Air Temperature [K]** — 0.0713
5. **Process Temperature [K]** — 0.0377

For each equipment record, SHAP was also used to identify the feature with the strongest contribution to the model prediction and whether that contribution increased or decreased the predicted failure risk.

SHAP explains the behavior of the machine learning model and should not be interpreted as proving physical causation.

In [16]:
import shap
import pandas as pd
import numpy as np

# --------------------------------------------------
# SHAP Explainability - Random Forest
# --------------------------------------------------

# Create SHAP explainer for the trained Random Forest
explainer = shap.TreeExplainer(rf_model)

# Calculate SHAP values for the test set
shap_values = explainer.shap_values(X_test)

# Handle SHAP output format
if isinstance(shap_values, list):
    # Older SHAP format: one array per class
    failure_shap_values = shap_values[1]
else:
    # Newer SHAP format
    if shap_values.ndim == 3:
        failure_shap_values = shap_values[:, :, 1]
    else:
        failure_shap_values = shap_values

# --------------------------------------------------
# Global Feature Importance
# Mean absolute SHAP value for each feature
# --------------------------------------------------

mean_abs_shap = np.abs(failure_shap_values).mean(axis=0)

shap_importance = pd.DataFrame({
    "Feature": X_test.columns,
    "Mean Absolute SHAP": mean_abs_shap
}).sort_values(
    by="Mean Absolute SHAP",
    ascending=False
)

print("=" * 55)
print("SHAP - GLOBAL FEATURE IMPORTANCE")
print("=" * 55)

print(shap_importance.to_string(index=False))


# --------------------------------------------------
# Top prediction driver for each test record
# --------------------------------------------------

top_driver_index = np.abs(failure_shap_values).argmax(axis=1)

top_risk_drivers = [
    X_test.columns[i]
    for i in top_driver_index
]

top_driver_shap = failure_shap_values[
    np.arange(len(X_test)),
    top_driver_index
]

top_driver_direction = np.where(
    top_driver_shap > 0,
    "Increases Failure Prediction",
    "Decreases Failure Prediction"
)

print("\nSHAP analysis completed successfully.")

SHAP - GLOBAL FEATURE IMPORTANCE
                Feature  Mean Absolute SHAP
            Torque [Nm]            0.186017
 Rotational speed [rpm]            0.131683
        Tool wear [min]            0.097697
    Air temperature [K]            0.071287
Process temperature [K]            0.037711

SHAP analysis completed successfully.


## Power BI Data Export

The outputs from the predictive model, anomaly detection, and SHAP analysis were combined into a single structured dataset for visualization and monitoring in Power BI.

The exported dataset includes:

- Operational sensor measurements
- Actual machine failure labels
- Failure probabilities
- Predicted failure classifications
- Risk levels
- Anomaly detection results and scores
- Top SHAP prediction drivers
- SHAP contribution values and directions

This dataset supports an interactive Power BI dashboard for monitoring equipment risk, model predictions, anomalous behavior, and prediction drivers.

In [17]:
# --------------------------------------------------
# Final Results Export
# Random Forest + Isolation Forest + SHAP
# --------------------------------------------------

final_results = X_test.copy()

# Random Forest results
final_results["Actual Failure"] = y_test.values
final_results["Failure Probability"] = rf_test_prob
final_results["Predicted Failure"] = rf_test_pred

# Risk Level
final_results["Risk Level"] = np.where(
    final_results["Failure Probability"] > best_rf_threshold,
    "High Risk",
    np.where(
        final_results["Failure Probability"] >= best_rf_threshold / 2,
        "Medium Risk",
        "Low Risk"
    )
)

# Isolation Forest results
final_results["Anomaly Status"] = anomaly_status
final_results["Anomaly Score"] = anomaly_scores

# SHAP explainability
final_results["Top Risk Driver"] = top_risk_drivers
final_results["Top Driver SHAP"] = top_driver_shap
final_results["Top Driver Direction"] = top_driver_direction

# Save CSV
output_file = "predictive_maintenance_results_final.csv"

final_results.to_csv(
    output_file,
    index=False
)

print("=" * 55)
print("FINAL DATASET CREATED")
print("=" * 55)

print("File:", output_file)
print("Rows:", final_results.shape[0])
print("Columns:", final_results.shape[1])

print("\nColumns:")
print(final_results.columns.tolist())

print("\nPreview:")
display(final_results.head())

FINAL DATASET CREATED
File: predictive_maintenance_results_final.csv
Rows: 1500
Columns: 14

Columns:
['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Actual Failure', 'Failure Probability', 'Predicted Failure', 'Risk Level', 'Anomaly Status', 'Anomaly Score', 'Top Risk Driver', 'Top Driver SHAP', 'Top Driver Direction']

Preview:


,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Actual Failure,Failure Probability,Predicted Failure,Risk Level,Anomaly Status,Anomaly Score,Top Risk Driver,Top Driver SHAP,Top Driver Direction
1529,298.1,309.0,1557,36.0,56,0,0.000,0,Low Risk,Normal,0.083551,Torque [Nm],-0.189781,Decreases Failure Prediction
8572,297.7,308.5,1559,35.2,127,0,0.000,0,Low Risk,Normal,0.075071,Torque [Nm],-0.187381,Decreases Failure Prediction
4330,301.5,309.8,1549,38.5,81,0,0.000,0,Low Risk,Normal,0.080702,Torque [Nm],-0.199955,Decreases Failure Prediction
1502,298.0,308.7,1462,48.2,204,0,0.005,0,Low Risk,Normal,0.030845,Rotational speed [rpm],-0.187166,Decreases Failure Prediction
9355,298.3,308.9,1708,28.0,42,0,0.000,0,Low Risk,Normal,0.020131,Torque [Nm],-0.198266,Decreases Failure Prediction


## Conclusion

This project demonstrates an end-to-end machine learning workflow for predictive maintenance and equipment condition monitoring.

The final system combines:

- **Random Forest** for machine failure prediction
- **Validation-based threshold optimization** for imbalanced classification
- **Isolation Forest** for unsupervised anomaly detection
- **SHAP** for model explainability
- **Power BI** for interactive monitoring and visualization

The final Random Forest model achieved an **F1 Score of 68.69%** and a **PR-AUC of 0.761** on the held-out test set.

The project demonstrates how machine learning, anomaly detection, explainable AI, and data visualization can be integrated into a practical predictive maintenance workflow.

### Limitations and Future Work

The dataset is useful for developing and evaluating the machine learning pipeline, but it contains a limited number of failure cases and a relatively small set of operational features.

Future work could include validation on larger equipment-specific datasets, additional sensor measurements, time-series data, and further model optimization for real industrial environments.